# Question B1 (15 marks)

> Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

> PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

> For questions B1 and B2, the following features should be used:   
> - **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
> - **Categorical** features: month, town, flat_model_type, storey_range

In [1]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)



---



## 1. Divide Dataset into Train, Validation, Test

> Divide the dataset (‘hdb_price_prediction.csv’) into train, validation and test sets by using entries from year 2019 and before as training data, year 2020 as validation data and year 2021 as test data.
**Do not** use data from year 2022 and year 2023.



In [2]:
df = pd.read_csv('hdb_price_prediction.csv')

# Define the columns to be used
cont_cols = [
    "dist_to_nearest_stn"       , 
    "dist_to_dhoby"             ,     
    "degree_centrality"         , 
    "eigenvector_centrality"    , 
    "remaining_lease_years"     , 
    "floor_area_sqm"
]

cat_cols = [
    "month"             , 
    "town"              ,  
    "flat_model_type"   , 
    "storey_range"
]

target_cols = [
    "resale_price"
]

train   = df[df["year"] <= 2019].copy()
val     = df[df["year"] == 2020].copy()
test    = df[df["year"] == 2021].copy()

In [3]:
print (f"Training Data Shape: {train.shape}")
print (f"Validation Data Shape: {val.shape}")
print (f"Test Data Shape: {test.shape}")

Training Data Shape: (64057, 14)
Validation Data Shape: (23313, 14)
Test Data Shape: (29057, 14)


In [4]:
# Print mean and std of the target variable in the different splits
print("\nTarget Variable Statistics:")
print(f"Train - Mean: {train[target_cols[0]].mean():.2f}, Std: {train[target_cols[0]].std():.2f}")
print(f"Validation - Mean: {val[target_cols[0]].mean():.2f}, Std: {val[target_cols[0]].std():.2f}")
print(f"Test - Mean: {test[target_cols[0]].mean():.2f}, Std: {test[target_cols[0]].std():.2f}")



Target Variable Statistics:
Train - Mean: 438923.16, Std: 153953.72
Validation - Mean: 452312.74, Std: 154533.74
Test - Mean: 511417.39, Std: 162643.36


In [5]:
# Print a table of for each categorical column (as row), and for each split (as column), the number of unique values and the top 3 most frequent values with their counts
print("\nCategorical Column Statistics:")
for col in cat_cols:
    print(f"\nColumn: {col}")
    for split_name, split_df in zip(["Train", "Validation", "Test"], [train, val, test]):
        unique_values = split_df[col].unique()
        num_unique = len(unique_values)
        top_3 = split_df[col].value_counts().head(3)
        top_3_str = ", ".join([f"{val} ({count})" for val, count in top_3.items()])
        print(f"{split_name} - Unique Values: {num_unique}, Top 3: {top_3_str}")


Categorical Column Statistics:

Column: month
Train - Unique Values: 12, Top 3: 7 (6412), 10 (5963), 8 (5906)
Validation - Unique Values: 12, Top 3: 9 (2481), 12 (2480), 7 (2453)
Test - Unique Values: 12, Top 3: 8 (2735), 7 (2655), 11 (2566)

Column: town
Train - Unique Values: 26, Top 3: WOODLANDS (4979), SENGKANG (4947), JURONG WEST (4934)
Validation - Unique Values: 26, Top 3: SENGKANG (2157), PUNGGOL (1745), TAMPINES (1700)
Test - Unique Values: 26, Top 3: SENGKANG (2749), PUNGGOL (2735), TAMPINES (2148)

Column: flat_model_type
Train - Unique Values: 41, Top 3: 4 ROOM, Model A (15536), 5 ROOM, Improved (11019), 3 ROOM, New Generation (5815)
Validation - Unique Values: 42, Top 3: 4 ROOM, Model A (5773), 5 ROOM, Improved (4252), 3 ROOM, New Generation (1749)
Test - Unique Values: 42, Top 3: 4 ROOM, Model A (7416), 5 ROOM, Improved (5286), 3 ROOM, New Generation (1961)

Column: storey_range
Train - Unique Values: 17, Top 3: 04 TO 06 (14834), 07 TO 09 (13635), 10 TO 12 (12234)
Valida

`flat_model_type` has an unseen category in validation and test

---

## 2. Model

> Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
> - Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
> - Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
> - Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
> - Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
> - Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

### 2.1 `DataConfig`

`DataConfig` in `PyTorch Tabular` defines how the data is managed, prerprocessed, and fed into the model

In [6]:
data_config = DataConfig(
    target                          = target_cols,  # target column name
    continuous_cols                 = cont_cols ,   # list of continuous column names to use
    categorical_cols                = cat_cols  ,   # list of categorical column names to use
    normalize_continuous_features   = True          # whether to normalize continuous features
) 

### 2.2 `TrainerConfig`

`TrainerConfig` in `PyTorch Tabular` manages the training process

In [7]:
trainer_config = TrainerConfig(
    auto_lr_find    = True,   # automatically tune learning rate
    batch_size      = 1024,
    max_epochs      = 50,
    seed            = SEED,
)

### 2.3 `CategoryEmbeddingModelConfig`

`CategoryEmbeddingModelConfig` in `PyTorch Tabular` defines the Feed-Foward Multi-Layer Perceptron

In [8]:
# Define the Model Architecture
model_config = CategoryEmbeddingModelConfig(
    task        = "regression",
    layers      = "50",     # one hidden layer with 50 neurons
    activation  = "ReLU",
)

### 2.4 `OptimizerConfig`

`OptimizerConfig` in `PyTorch Tabular` defines the optimizer used for training 

In [9]:
optimizer_config = OptimizerConfig(
    optimizer="Adam"
)

### 2.5 `TabularModel`

`TabularModel`in `PyTorch Tabular` combines all of previous configurations to create a unified pipeline for training and testing

In [10]:
tabular_model = TabularModel(
    data_config         = data_config       ,
    model_config        = model_config      ,
    optimizer_config    = optimizer_config  ,
    trainer_config      = trainer_config,
)

2026-03-05 23:03:44,626 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


---

## 3. Train

> Report the test MSE error and the test R2 value that you obtained.



**Global Monkey Patch**

PyTorch 2.6+ now defaults to `weights_only=True` in `torch.load()` for security. This prevents the loading of custom objects or certain module types that are not in the default allowlist to protect against arbitrary code execution.

So we force all `torch.load` calls to use `weights_only=False` by overwriting the function at the start of the script

In [11]:
import torch

# Capture the original function
_orig_torch_load = torch.load

# Define a version that always sets weights_only to False
def _trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)

# Overwrite the global torch.load
torch.load = _trusted_load

**Why `StandardScaler` on target?**

Previously we only applied `StandardScaler` (or its encompassing method `preprocess_dataset` only features only (i.e. `X_train`, `X_test`)

However, our target here are in the range of 100 thousands (big mean and large variance). Most NN starts with weights around 0 and predict in single digits too (e.g. 0.2, 3). When the starting prediction is too far away from target, numeric instability leads to the model unable to update its weight correctly.

Thus, we scale the target to be in similar range (single digits). PyTorch Tabular is smart enough to run `scaler.inverse_transform()` on the final output to convert the predictions back to its original scale.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler

# Define your scaler
target_scaler = StandardScaler()

# Train the model using train/validation split
tabular_model.fit(
    train=train, 
    validation=val,
    target_transform=target_scaler
)

# Predict on test set
test_predictions = tabular_model.predict(test)

test_predictions.head()

Seed set to 42
2026-03-05 23:03:44,639 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-05 23:03:44,651 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-05 23:03:44,687 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-05 23:03:44,706 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-05 23:03:44,778 - {pytorch_tabular.tabular_model:655} - INFO - Auto LR Find Started
/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/leyanzhi

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /Users/leyanzhi/Repositories/SC4001-Neural-Network-Assignment/.lr_find_e37190f1-79a5-4a4f-a3c9-b37de7ccc4f0.ckpt
Restored all states from the checkpoint at /Users/leyanzhi/Repositories/SC4001-Neural-Network-Assignment/.lr_find_e37190f1-79a5-4a4f-a3c9-b37de7ccc4f0.ckpt
Learning rate set to 9.120108393559096e-06
2026-03-05 23:03:55,313 - {pytorch_tabular.tabular_model:668} - INFO - Suggested LR: 9.120108393559096e-06. For plot and detailed analysis, use `find_learning_rate` method.
2026-03-05 23:03:55,313 - {pytorch_tabular.tabular_model:677} - INFO - Training Started


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-05 23:04:55,835 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-05 23:04:55,835 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


,resale_price_prediction
87370,297881.31250
87371,309271.75000
87372,333522.71875
87373,329775.81250
87374,320321.18750


**MSE (Mean Squared Error)**

Average squared difference between actual values and predicted values (sqaured to penalize larger errors more heavily than smaller ones)
- Between 0 to infinity (0 indicates perfect fit)

**R-squared (Coeffieicent of Determination)**

Measures proportion of variance in the dependent variable that is explained by the model. </br>
Indicated the goodness of fit of the model compared to a simple horizon line (the mean of the data)
- Usually between 0 and 1 
- 1 indicates perfect fit (model explains all variability)
- 0 indicates the model is no better than just predicting the mean of the data (model explains none of the variability)
- negative indicates the model is worse than just prediciting the mean

In [13]:
# Get prediction column (usually "resale_price_prediction")
predicted_cols = [col for col in test_predictions.columns if col.endswith("_prediction")]
predicted_col = predicted_cols[0]  # one target column

y_true = test[target_cols[0]].to_numpy()
y_pred = test_predictions[predicted_col].to_numpy()

# Metrics
test_mse = mean_squared_error(y_true, y_pred)
test_r2 = r2_score(y_true, y_pred)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2 : {test_r2:.4f}")

Test MSE: 19619217627.9618
Test R2 : 0.2583


---

## 4. Top 20 Test Samples with Largest Errors

> Print out the corresponding rows in the dataframe for the top 20 test samples with the largest errors. Are there any patterns or common characteristics among these data points? Based on your observations, suggest and briefly explain a potential strategy to improve the model on these cases.



In [14]:
# We use abs error for comparison
error_df = test.copy()
error_df["resale_price_prediction"] = test_predictions[predicted_col]
error_df["abs_error"] = (error_df["resale_price"] - error_df["resale_price_prediction"]).abs()

# Top 20 test samples with largest absolute errors
top20_largest_errors = error_df.sort_values("abs_error", ascending=False).head(20)

top20_largest_errors

,month,year,town,full_address,nearest_stn,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,flat_model_type,remaining_lease_years,floor_area_sqm,storey_range,resale_price,resale_price_prediction,abs_error
90608,12,2021,BISHAN,273B BISHAN STREET 24,Bishan,0.776182,6.297489,0.033613,0.015854,"5 ROOM, DBSS",88.833333,120.0,37 TO 39,1360000.0,470251.78125,889748.21875
106199,12,2021,QUEENSTOWN,92 DAWSON ROAD,Queenstown,0.584731,3.882019,0.016807,0.008342,"5 ROOM, Premium Apartment Loft",93.333333,122.0,40 TO 42,1328000.0,458999.93750,869000.06250
90483,9,2021,BISHAN,273A BISHAN STREET 24,Bishan,0.767244,6.327956,0.033613,0.015854,"5 ROOM, DBSS",89.000000,120.0,37 TO 39,1295000.0,500920.15625,794079.84375
101237,11,2021,KALLANG/WHAMPOA,8 BOON KENG ROAD,Bendemeer,0.352251,2.587444,0.016807,0.004414,"5 ROOM, DBSS",88.250000,119.0,40 TO 42,1268000.0,485721.18750,782278.81250
100836,6,2021,KALLANG/WHAMPOA,39 JALAN BAHAGIA,Boon Keng,0.998313,3.304953,0.016807,0.053004,"3 ROOM, Terrace",50.083333,210.0,01 TO 03,1268000.0,495665.06250,772334.93750
93931,12,2021,CENTRAL AREA,1B CANTONMENT ROAD,Outram Park,0.352779,2.413099,0.033613,0.121082,"5 ROOM, Type S2",88.083333,107.0,40 TO 42,1288000.0,524141.00000,763859.00000
90432,8,2021,BISHAN,275A BISHAN STREET 24,Bishan,0.827889,6.370404,0.033613,0.015854,"5 ROOM, DBSS",88.916667,120.0,25 TO 27,1280000.0,520739.37500,759260.62500
93904,11,2021,CENTRAL AREA,1C CANTONMENT ROAD,Outram Park,0.401367,2.445314,0.033613,0.121082,"5 ROOM, Type S2",88.333333,106.0,40 TO 42,1261000.0,517515.00000,743485.00000
90253,4,2021,BISHAN,273B BISHAN STREET 24,Bishan,0.776182,6.297489,0.033613,0.015854,"5 ROOM, DBSS",89.416667,120.0,31 TO 33,1250000.0,511408.53125,738591.46875
101236,11,2021,KALLANG/WHAMPOA,9 BOON KENG ROAD,Bendemeer,0.335875,2.535679,0.016807,0.004414,"5 ROOM, DBSS",88.250000,119.0,34 TO 36,1230000.0,492918.25000,737081.75000


**R2 = 0.2583** which isn't very good. This is because from data analysis earlier, mean resale price of 2021 (Test) is much higher than 2020 (validation) and significantly higher than 2019 and earlier (train). The resale price surged post-covid, and the model, trained only with pre-covid data, has difficulty forecasting this.

The top 20 test samples with largest error are all HDBs in CCR and OCR (no RCR actually), suggesting that more central HDBs faced higher surge in resale price post-covid.

Moreover, thet are mostly HDBs with large floor area and located in upper levels. This shows that luxrious HDBs are more favoured by buyers post-covid, as circuit breakers has shown the importance of a comfortable home for work and living.

To improve it, maybe we can fit the target of validation and test as well (using `Standard Scaler`), using the train target' scale. This way, the mean and distribution of the price of train, validation, test will be roughly the same and the model can ignore the effect of inflation when doing prediction.